# scRNA-seq 분석 실습 — QC부터 Cell Type Annotation까지

**건국대학교 이형우 교수님 연구실 온라인 세미나**
Data: GSE210543 (Human Retina — Young vs Old)

---

### 세미나 커리큘럼 (총 3시간)

| Step | 내용 | 예상시간 |
|------|------|---------|
| Step 0 | 환경 설정 (패키지 설치) | 10분 |
| Step 1 | 데이터 불러오기 | 10분 |
| Step 2 | QC 이론 + 시각화 | 30분 |
| Step 3 | QC 필터링 | 10분 |
| Step 4 | 정규화 (Normalization) | 10분 |
| Step 5 | HVG 선택 | 5분 |
| **—** | **중간 체크포인트 저장** | — |
| Step 6 | Cell Cycle (optional) | 5분 |
| Step 7 | Scaling + PCA | 10분 |
| Step 8 | Integration (Harmony) | 10분 |
| Step 9 | UMAP | 10분 |
| Step 10 | Clustering | 15분 |
| Step 11 | Marker Identification | 15분 |
| Step 12 | Cell Type Annotation | 20분 |

### 분석 샘플
| 그룹 | 샘플 | 세포 수 |
|------|------|--------|
| Young (발달기) | 16PCW | 8,601 |
| Young (발달기) | 20PCW | 5,787 |
| Old (성체) | Adult_2 | 6,516 |
| Old (성체) | Adult_3 | 3,694 |

---
## Background. 데이터셋 개요 — GSE210543

### Cell Ranger Web Summary란?

10X Genomics Cell Ranger 파이프라인이 시퀀싱 완료 후 생성하는 **QC 리포트**입니다.
분석 시작 전 샘플 품질을 가장 먼저 확인하는 파일입니다.

주요 지표:
- **Estimated Number of Cells**: Cell Ranger가 추정한 세포 수
- **Mean Reads per Cell**: 세포당 평균 시퀀싱 reads 수
- **Median Genes per Cell**: 세포당 검출된 유전자 수 중앙값 (샘플 품질의 핵심 지표)
- **Sequencing Saturation**: 시퀀싱 포화도 (높을수록 재시퀀싱 효과 낮음)

> **Tip**: Median Genes per Cell이 낮으면 포획된 RNA 양이 적거나 세포 품질이 낮을 수 있습니다.

### 전체 13개 샘플 품질 요약

| 샘플 | 그룹 | 세포 수 | Mean Reads/Cell | Median Genes/Cell | 선택 여부 |
|------|------|-------:|---------------:|------------------:|:--------:|
| **16PCW** | Young | **8,601** | 43,243 | 1,084 | ✅ 선택 |
| **20PCW** | Young | **5,787** | 103,036 | **5,174** | ✅ 선택 |
| 12PCW | Young | 3,637 | 164,166 | 4,596 | — |
| 21PCW | Young | 3,948 | 139,139 | 1,953 | — |
| **Adult_2** | Old | **6,516** | 90,897 | 2,016 | ✅ 선택 |
| **Adult_3** | Old | **3,694** | 142,303 | 2,806 | ✅ 선택 |
| Adult_5 | Old | 3,487 | 168,631 | 2,529 | — |
| Adult_1 | Old | 884 | 412,206 | 2,807 | ❌ 세포 수 부족 |
| Adult_4 | Old | 496 | 1,146,570 | 2,414 | ❌ 세포 수 부족 + 비정상 reads |
| AMD_macula | AMD | 1,719 | 284,749 | 4,002 | — (연구 목적 외) |
| AMD_peripheral | AMD | 1,794 | 267,813 | 3,584 | — (연구 목적 외) |
| Unaffected_macula | Control | 3,557 | 101,143 | 5,102 | — (연구 목적 외) |
| Unaffected_peripheral | Control | 3,527 | 112,731 | 4,765 | — (연구 목적 외) |

### 4개 샘플 선택 이유

**연구 목적: Young (발달기) vs Old (성체) 비교**

| 구분 | 이유 |
|------|------|
| AMD/Unaffected 제외 | 질환 연구 목적 — 이번 세미나의 비교 목적과 다름 |
| Adult_1, Adult_4 제외 | 세포 수 < 1,000개 (통계적 분석에 부적절) |
| Adult_4 추가 제외 | Mean Reads 1,146,570 — 극소수 세포에 reads 편중, 라이브러리 품질 문제 의심 |
| 12PCW, 21PCW 미선택 | 16PCW, 20PCW와 중복 발달 시기, 세포 수 우세한 샘플 우선 선택 |
| Adult_5 미선택 | Adult_2, Adult_3 대비 세포 수 적음 |

---
### Web Summary 읽는 법 — 좋은 샘플 vs 나쁜 샘플 비교

분석 시작 전 Web Summary로 샘플 품질을 판별하는 법을 알아봅니다.

**체크해야 할 핵심 지표 3가지:**
| 지표 | 의미 | 건강한 범위 |
|------|------|-----------|
| Estimated Number of Cells | Cell Ranger가 추정한 세포 수 | > 2,000 권장 |
| Median Genes per Cell | 세포당 검출 유전자 수 중앙값 | > 1,500 (조직별 상이) |
| Fraction Reads in Cells | 세포로 할당된 reads 비율 | > 70% |

> **Mean Reads per Cell**이 매우 높다면? → 세포가 너무 적기 때문에 생기는 수학적 효과일 수 있습니다. 단독으로 해석하지 마세요!

#### ✅ 좋은 샘플 — 20PCW (임신 20주 태아 망막)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_good_20PCW.png" width="900"/>

**20PCW 샘플 분석:**

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Number of Cells | **5,787** | ✅ 충분한 세포 수 |
| Mean Reads per Cell | 103,036 | ✅ 합리적 (총 reads ÷ 세포 수) |
| Median Genes per Cell | **5,174** | ✅ 매우 높음 — RNA 포획 우수 |
| Fraction Reads in Cells | **91.3%** | ✅ 배경 노이즈 최소 |

**Barcode Rank Plot 해석:** 파란 선(Cells)과 회색 선(Background) 사이에 뚜렷한 **꺾임(knee/inflection point)**이 보입니다. 이 꺾임이 명확할수록 세포와 빈 droplet의 구분이 잘 됩니다.

→ 이 샘플은 **세미나 분석 대상**으로 선택했습니다.

#### ❌ 나쁜 샘플 — Adult_4 (성체 망막)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_bad_Adult4.png" width="900"/>

**Adult_4 샘플 분석:**

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Number of Cells | **496** | ❌ 극히 적음 — 세포 포획 실패 의심 |
| Mean Reads per Cell | **1,146,570** | ⚠️ 비정상적으로 높음 |
| Median Genes per Cell | 2,414 | ⚠️ 세포 수 대비 보통 |
| Fraction Reads in Cells | 78.8% | ⚠️ 낮은 편 |

**왜 Mean Reads가 1,146,570으로 폭등했을까?**

```
Mean Reads per Cell = 총 시퀀싱 reads ÷ 검출된 세포 수
                    = 568,698,782 ÷ 496
                    ≈ 1,146,570
```

총 reads 수는 20PCW(596M)와 비슷하지만, 세포 수가 5,787개 → 496개로 급감했기 때문입니다.
**세포 포획 실패(capture failure)** — 대부분의 reads가 소수의 세포에 몰린 상태.

**Barcode Rank Plot 해석:** x축이 10k에서 끝남 → 전체적으로 포획된 바코드 수가 매우 적습니다. 정상 샘플(20PCW)은 1M까지 연장됩니다.

→ 이 샘플은 **통계 분석 불가** (세포 수 < 1,000) — 분석에서 제외했습니다.

In [ ]:
# Web Summary 파일 열기 (Google Drive에서)
# 아래 경로의 HTML 파일을 브라우저에서 직접 열어 확인하세요

WEB_SUMMARY_DIR <- "/content/drive/MyDrive/KU_seminar/web_summary"

# 파일 목록 확인
ws_files <- list.files(WEB_SUMMARY_DIR, pattern = "*.html", full.names = FALSE)
cat("Web Summary 파일 목록:\n")
print(ws_files)

# Colab에서 HTML 파일 미리보기 (선택한 4개 샘플)
selected_samples <- c("16PCW", "20PCW", "Adult_2", "Adult_3")
for (s in selected_samples) {
  f <- file.path(WEB_SUMMARY_DIR, paste0(s, "_web_summary.html"))
  if (file.exists(f)) {
    cat(sprintf("\n%s web summary: %s\n", s, f))
  }
}
cat("\n위 경로의 HTML 파일을 Google Drive에서 직접 열어보세요.\n")

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_04.png" width="850"/>

*Fig. 0 — scRNA-seq 분석의 복잡성 (Hicks et al., BioRxiv 2015)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_05.png" width="850"/>

*Fig. 1 — scRNA-seq 전체 분석 워크플로우 (Luecken & Theis, Mol Syst Biol 2019)*

---
## Step 0. 환경 설정 (~10분)

> **Colab R 런타임 설정 확인**: Runtime → Change runtime type → **R**
>
> 패키지 설치는 처음 1번만. 세션이 끊기면 재실행 필요 (~10분 소요)

In [ ]:
# 필요 패키지 설치 (처음 1회만 실행 — 약 10분 소요)
pkgs <- c("Seurat", "harmony", "dplyr", "ggplot2", "patchwork", "clustree", "plyr")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p)
}

In [ ]:
library(Seurat)
library(harmony)
library(dplyr)
library(ggplot2)
library(patchwork)

set.seed(42)
cat("Seurat:", as.character(packageVersion("Seurat")), "\n")
cat("harmony:", as.character(packageVersion("harmony")), "\n")

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 scRNA-seq 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1: 사용자 환경에 맞게 선택하라*

---
## Step 1. Google Drive 연결 및 데이터 불러오기 (~10분)

### 데이터 구조 (10X Genomics filtered_feature_bc_matrix)

```
sample_folder/
├── barcodes.tsv.gz   ← 세포 바코드 목록 (행 = 세포)
├── features.tsv.gz   ← 유전자 목록 (열 = 유전자)
└── matrix.mtx.gz     ← 발현량 행렬 (sparse matrix)
```

> **10X Genomics 작동 원리**: 각 세포를 액적(droplet)에 가두고 고유한 바코드를 부여합니다.  
> mRNA는 역전사 후 시퀀싱되며, **UMI(Unique Molecular Identifier)**로 PCR 중복을 제거합니다.

In [ ]:
# Google Drive 마운트
# Colab 좌측 파일 아이콘 → Drive 마운트 버튼 클릭
# 또는 아래 실행 후 인증 링크 클릭

# R에서 Drive 마운트 명령 실행
system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# 데이터 경로 설정 (공유된 Google Drive 경로로 수정 필요)
DATA_DIR <- "/content/drive/MyDrive/KU_seminar/GSE210543/filtered_feature_bc_matrix"
cat("Data directory:", DATA_DIR, "\n")

In [ ]:
# 4개 샘플 불러오기
samples <- list(
  Young_16PCW  = file.path(DATA_DIR, "16PCW"),
  Young_20PCW  = file.path(DATA_DIR, "20PCW"),
  Old_Adult2   = file.path(DATA_DIR, "Adult_2"),
  Old_Adult3   = file.path(DATA_DIR, "Adult_3")
)

# Seurat 오브젝트 생성
seurat_list <- lapply(names(samples), function(name) {
  cat("Loading:", name, "...\n")
  counts <- Read10X(data.dir = samples[[name]])
  obj <- CreateSeuratObject(
    counts  = counts,
    project = name,
    min.cells = 3,    # 최소 3개 세포에서 발현된 유전자만 포함
    min.features = 200  # 최소 200개 유전자가 발현된 세포만 포함
  )
  obj$sample <- name
  obj$group  <- ifelse(grepl("Young", name), "Young", "Old")
  return(obj)
})
names(seurat_list) <- names(samples)

# 각 샘플 기본 정보 확인
for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat(sprintf("%s: %d cells, %d genes\n", name, ncol(obj), nrow(obj)))
}

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_08.png" width="850"/>

*Fig. 4 — Count Matrix 생성 원리: barcodes / features / matrix (Macosko et al., Cell 2015)*

---
## Step 2. QC — Quality Control (~30분)

### 🧬 QC 지표의 생물학적 의미

scRNA-seq 데이터에는 **저품질 세포(low-quality cells)**가 섞여 있습니다.  
이를 제거하지 않으면 분석 결과가 왜곡됩니다.

---

#### 1️⃣ nFeature_RNA — 세포당 검출된 유전자 수

```
정상 세포:  200 ~ 6,000개 유전자
빈 액적:    < 200개    → 세포가 없는 빈 droplet
이중 세포:  매우 높음   → 두 세포가 하나로 잡힌 doublet
```

#### 2️⃣ nCount_RNA (nUMI) — 세포당 전체 UMI 수

```
정상 세포:  500 ~ 50,000 UMI
저품질:    < 500       → RNA 포획 실패 또는 세포 사멸
이중 세포:  매우 높음   → doublet 의심
```

> **UMI(Unique Molecular Identifier)**: PCR 증폭 편향을 보정하는 고유 분자 바코드.  
> 동일한 UMI를 가진 read는 같은 분자에서 온 것으로 간주하여 1개로 계수합니다.

#### 3️⃣ percent.mt — 미토콘드리아 유전자 비율

```
정상 세포:  < 10%
손상/사멸:  > 10%  → 세포막 손상 시 세포질 RNA가 빠져나가지만
                      미토콘드리아는 막으로 둘러싸여 있어 RNA가 남음
```

> **왜 미토콘드리아인가?**  
> 세포가 죽거나 스트레스를 받으면 세포질의 RNA가 용해되어 빠져나갑니다.  
> 하지만 미토콘드리아는 자체 막을 가지고 있어 RNA가 남아있습니다.  
> 따라서 mt% 비율이 높을수록 손상된 세포를 의미합니다.

---

### 📊 이번 세미나 QC 기준 (Default Threshold)

| 지표 | 기준 | 의미 |
|------|------|------|
| nFeature_RNA | **> 200** | 최소 200개 유전자 발현 세포만 유지 |
| nCount_RNA | **> 500** | 최소 500 UMI 이상 세포만 유지 |
| percent.mt | **< 10%** | 미토콘드리아 비율 10% 미만 |

> **Tip**: 이 기준은 고정된 것이 아닙니다. 조직 유형, 실험 조건에 따라 조정이 필요합니다.  
> 시각화 후 데이터 분포를 보고 결정하는 것이 좋습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_09.png" width="850"/>

*Fig. 5 — QC 지표 분포 확인 방법 (NCells, nUMI, nGene, mitoRatio)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_10.png" width="850"/>

*Fig. 6 — QC VlnPlot 예시: nFeature_RNA / nCount_RNA / percent.mt*

In [ ]:
# 미토콘드리아 유전자 비율 계산
# 인간 미토콘드리아 유전자는 'MT-' 로 시작
seurat_list <- lapply(seurat_list, function(obj) {
  obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
  return(obj)
})

# QC 지표 확인 (첫 번째 샘플 예시)
head(seurat_list[[1]]@meta.data[, c("nFeature_RNA", "nCount_RNA", "percent.mt")])

In [ ]:
# QC 지표 시각화 — Violin Plot
# 각 샘플별로 분포를 확인합니다

plot_list <- lapply(names(seurat_list), function(name) {
  VlnPlot(
    seurat_list[[name]],
    features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
    ncol = 3,
    pt.size = 0
  ) + plot_annotation(title = name)
})

# 샘플별 출력
for (p in plot_list) print(p)

In [ ]:
# nFeature vs nCount 산점도 — doublet 탐지
# 정상 세포는 선형 관계를 보임
# 이상치(doublet)는 오른쪽 상단에 위치

scatter_list <- lapply(names(seurat_list), function(name) {
  p1 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "percent.mt"
  ) + ggtitle(paste(name, "- Count vs MT%"))

  p2 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "nFeature_RNA"
  ) + ggtitle(paste(name, "- Count vs Feature"))

  p1 + p2
})

for (p in scatter_list) print(p)

### 🔍 QC 분포 요약 통계

시각화 후, 각 샘플의 분포를 수치로도 확인합니다.  
특히 **중앙값(median)**과 **이상치 범위**를 주목하세요.

In [ ]:
# 샘플별 QC 요약 통계
qc_summary <- do.call(rbind, lapply(names(seurat_list), function(name) {
  meta <- seurat_list[[name]]@meta.data
  data.frame(
    Sample         = name,
    Group          = unique(meta$group),
    Cells          = nrow(meta),
    nFeature_median = median(meta$nFeature_RNA),
    nFeature_max    = max(meta$nFeature_RNA),
    nCount_median   = median(meta$nCount_RNA),
    mt_median       = round(median(meta$percent.mt), 2),
    mt_max          = round(max(meta$percent.mt), 2)
  )
}))

print(qc_summary)

---
## Step 3. QC 필터링 적용 (~10분)

시각화로 분포를 확인한 후, 아래 기준으로 저품질 세포를 제거합니다.

| 지표 | 기준 |
|------|------|
| nFeature_RNA | **> 200** |
| nCount_RNA | **> 500** |
| percent.mt | **< 10%** |

> **💡 Discussion**: 위 VlnPlot을 보고 이 기준이 적절한지 토론해봅시다.  
> - 이 샘플에서 특이한 분포가 보이는 샘플이 있나요?
> - 더 엄격한 기준이 필요한 경우는 언제일까요?

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_11.png" width="850"/>

*Fig. 7 — QC 필터링 실습 코드 및 Tip 2: Filter 조건에서의 정답은 없다!*

In [ ]:
# QC 기준값 설정 (여기서 조정 가능)
QC_MIN_FEATURE <- 200   # 최소 유전자 수
QC_MIN_COUNT   <- 500   # 최소 UMI 수
QC_MAX_MT      <- 10    # 최대 미토콘드리아 비율 (%)

# 필터링 적용
seurat_list_filtered <- lapply(names(seurat_list), function(name) {
  obj <- seurat_list[[name]]
  before <- ncol(obj)

  obj <- subset(
    obj,
    subset = nFeature_RNA > QC_MIN_FEATURE &
             nCount_RNA   > QC_MIN_COUNT   &
             percent.mt   < QC_MAX_MT
  )

  after <- ncol(obj)
  removed <- before - after
  cat(sprintf("%s: %d → %d cells (removed %d, %.1f%%)\n",
              name, before, after, removed, removed/before*100))
  return(obj)
})
names(seurat_list_filtered) <- names(seurat_list)

In [ ]:
# 필터링 후 QC 분포 재확인
plot_list_after <- lapply(names(seurat_list_filtered), function(name) {
  VlnPlot(
    seurat_list_filtered[[name]],
    features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
    ncol = 3,
    pt.size = 0
  ) + plot_annotation(title = paste(name, "[After QC]"))
})

for (p in plot_list_after) print(p)

---
## Step 4. 정규화 — Normalization (~10분)

### 🧬 왜 정규화가 필요한가?

각 세포마다 포획된 RNA의 총량이 다릅니다.  
이를 보정하지 않으면 **RNA 포획 효율의 차이**가 마치 **생물학적 차이**처럼 보입니다.

#### LogNormalize 방법
```
normalized = log( (count / total_count_per_cell) × scale_factor + 1 )
```
- `scale_factor`: 기본값 10,000 (CPM과 유사)
- `log1p`: 0인 값을 처리하고 분포를 정규화

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_12.png" width="850"/>

*Fig. 8 — LogNormalization vs SCTransform 방법 비교 및 PC 선택 기준 (Tip 3)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_13.png" width="850"/>

*Fig. 9 — 정규화 개념: SCTransform vs LogNormalization (M. Loven, RNA-seq statistical analysis)*

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  NormalizeData(
    obj,
    normalization.method = "LogNormalize",
    scale.factor = 10000
  )
})

cat("Normalization complete!\n")

---
## Step 5. 고변이 유전자 선택 — HVG (~5분) (Highly Variable Genes, HVG)

### 🧬 왜 HVG를 선택하는가?

인간 게놈의 약 20,000개 유전자 중 대부분은 **모든 세포에서 비슷하게 발현**됩니다.  
세포 유형을 구분하는 데 유용한 유전자는 **세포마다 발현량이 크게 다른 유전자**입니다.

- 너무 적게 발현: 노이즈가 많음
- 모든 세포에서 균일하게 발현: 정보 없음
- **세포마다 다르게 발현**: 세포 유형 구분에 유용 ✓

**기본값: 상위 2,000개 HVG 선택**

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  FindVariableFeatures(
    obj,
    selection.method = "vst",
    nfeatures = 2000
  )
})

# Top 10 HVG 확인 (첫 번째 샘플)
top10 <- head(VariableFeatures(seurat_list_filtered[[1]]), 10)
cat("Top 10 HVGs (16PCW):\n")
print(top10)

In [ ]:
# HVG 시각화
plot_hvg_list <- lapply(names(seurat_list_filtered), function(name) {
  obj  <- seurat_list_filtered[[name]]
  top10 <- head(VariableFeatures(obj), 10)
  p <- VariableFeaturePlot(obj)
  LabelPoints(plot = p, points = top10, repel = TRUE) +
    ggtitle(name)
})

for (p in plot_hvg_list) print(p)

---
## Step 6. 오브젝트 병합

QC와 정규화가 완료된 4개 샘플을 하나의 Seurat 오브젝트로 병합합니다.  
다음 파트(통합 분석)에서 사용합니다.

In [ ]:
# 4개 샘플 병합
seurat_merged <- merge(
  seurat_list_filtered[[1]],
  y    = seurat_list_filtered[2:4],
  add.cell.ids = names(seurat_list_filtered)
)

# 결과 확인
cat("Merged object:\n")
print(seurat_merged)
cat("\nSample composition:\n")
print(table(seurat_merged$sample))
cat("\nGroup composition:\n")
print(table(seurat_merged$group))

In [ ]:
# Google Drive에 저장 (다음 파트에서 로드 가능)
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
dir.create(SAVE_DIR, showWarnings = FALSE, recursive = TRUE)

saveRDS(seurat_merged, file = file.path(SAVE_DIR, "01_seurat_merged_after_QC.rds"))
cat("Saved:", file.path(SAVE_DIR, "01_seurat_merged_after_QC.rds"), "\n")

---
## 중간 체크포인트

QC + 정규화 + HVG까지 완료했습니다.
오브젝트를 저장해두고 이후 Integration부터 이어서 진행합니다.

> 세션이 끊기면 아래 로드 셀을 실행하면 됩니다.

In [ ]:
# 중간 저장
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
dir.create(SAVE_DIR, showWarnings = FALSE, recursive = TRUE)
saveRDS(seurat_merged, file.path(SAVE_DIR, "checkpoint_after_HVG.rds"))
cat("Saved checkpoint!\n")

In [ ]:
# (세션 재시작 시) 중간 저장본 불러오기
# — Step 0 라이브러리 로드 후 이 셀 실행 —
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
seurat_merged <- readRDS(file.path(SAVE_DIR, "checkpoint_after_HVG.rds"))
cat("Loaded checkpoint:", ncol(seurat_merged), "cells\n")

---
## Step 7. (Optional) Cell Cycle Scoring (~5분)

### 🔄 Cell Cycle을 고려해야 하는 이유

세포주기(G1/S/G2M)는 유전자 발현에 강한 영향을 미칩니다.
이를 보정하지 않으면 클러스터가 세포 타입이 아니라 **세포주기 상태**로 나뉠 수 있습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_14.png" width="850"/>

*Fig. 10 — Cell Cycle Scoring: 언제 회귀할지, 언제 남겨둘지*

> **세미나 데이터(망막)** 에서는 발달기 샘플(16PCW, 20PCW)에 증식 세포가 많습니다.
> 세포 타입 구분이 목적이므로 → **Cell Cycle 회귀 적용**

In [ ]:
# Cell cycle 관련 유전자 (Seurat 내장)
s.genes   <- cc.genes$s.genes    # S phase
g2m.genes <- cc.genes$g2m.genes  # G2M phase

# Cell cycle score 계산
seurat_merged <- CellCycleScoring(
  seurat_merged,
  s.features   = s.genes,
  g2m.features = g2m.genes,
  set.ident    = TRUE
)

# 분포 확인
table(seurat_merged$Phase)

---
## Step 8. Scaling + PCA (~10분)

### 📐 Scaling이 필요한 이유

유전자마다 발현 범위가 다릅니다(어떤 유전자는 0~2, 어떤 유전자는 0~1000).
Scaling은 각 유전자를 평균 0, 분산 1로 맞춰 **동등한 가중치**를 부여합니다.

```
scaled = (expression - mean) / sd
```

Cell cycle 효과를 회귀(regress out)해서 제거합니다.

In [ ]:
# Scaling (cell cycle 회귀 포함)
seurat_merged <- ScaleData(
  seurat_merged,
  vars.to.regress = c("S.Score", "G2M.Score"),  # cell cycle 보정
  features        = rownames(seurat_merged)
)

cat("Scaling complete!\n")

In [ ]:
# PCA 실행
seurat_merged <- RunPCA(
  seurat_merged,
  features = VariableFeatures(seurat_merged),
  npcs     = 50
)

# Elbow Plot — 몇 개의 PC를 사용할지 결정
ElbowPlot(seurat_merged, ndims = 50) +
  ggtitle("Elbow Plot: PC 기여도") +
  geom_vline(xintercept = 30, linetype = "dashed", color = "red") +
  annotate("text", x = 32, y = 3, label = "PC=30 선택", color = "red")

---
## Step 9. Integration — Harmony (~10분)

### 🧩 왜 Integration이 필요한가?

서로 다른 샘플(16PCW, 20PCW, Adult_2, Adult_3)은 각각 별도로 sequencing된 데이터입니다.
이 경우 **Batch Effect** — 기술적 변이 — 가 생물학적 차이처럼 보일 수 있습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_15.png" width="850"/>

*Fig. 11 — Integration 방법 비교: Harmony가 세포 타입 구조를 잘 보존 (Tran et al., Genome Biol 2020)*

> **Harmony 선택 이유**: 빠르고 메모리 효율적이며, scRNA-seq에서 검증된 방법
> Seurat v5에서는 `IntegrateLayers()` + `harmony` 사용

In [ ]:
# Seurat v5 방식: Layer-based integration
seurat_merged <- IntegrateLayers(
  object      = seurat_merged,
  method      = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction  = "harmony",
  group.by.vars  = "sample",  # 배치 변수: 샘플별 보정
  verbose     = FALSE
)

cat("Harmony integration complete!\n")

---
## Step 10. UMAP (~10분)

### 🗺️ UMAP이란?

UMAP(Uniform Manifold Approximation and Projection)은 고차원(50 PC) 데이터를 **2D로 시각화**하는 방법입니다.
가까운 세포 = 유사한 발현 패턴을 가진 세포

> **주의**: UMAP은 탐색적 시각화 도구입니다. 거리의 절대적 의미는 없습니다.

In [ ]:
# UMAP (Harmony 통합 결과 기반)
seurat_merged <- RunUMAP(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 통합 전/후 비교
p_before <- DimPlot(seurat_merged, reduction = "pca",  group.by = "sample") + ggtitle("Before Integration (PCA)")
p_after  <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample") + ggtitle("After Integration (UMAP)")
p_before + p_after

In [ ]:
# Young vs Old 분포 확인
DimPlot(seurat_merged, reduction = "umap", group.by = "group",
        cols = c("Young" = "#E67E22", "Old" = "#2980B9")) +
  ggtitle("UMAP: Young vs Old") +
  theme_minimal()

---
## Step 11. Clustering (~15분)

### 🔢 Graph-based Clustering

Seurat은 **KNN 그래프 + Louvain/Leiden 알고리즘**으로 세포를 클러스터링합니다.

**Resolution**: 클러스터 개수를 결정하는 핵심 파라미터
- Resolution ↑ → 클러스터 많아짐 (더 세분화)
- Resolution ↓ → 클러스터 적어짐 (더 뭉침)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_16.png" width="850"/>

*Fig. 12 — Resolution에 따른 클러스터링 결과 비교 (Res 0.2 ~ 1.2)*

> **Tip**: Resolution 0.4~0.6이 일반적으로 좋은 시작점
> 여러 Resolution 결과를 비교한 후 생물학적으로 의미 있는 것을 선택

In [ ]:
# KNN 그래프 생성
seurat_merged <- FindNeighbors(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 여러 Resolution으로 클러스터링
resolutions <- c(0.2, 0.4, 0.6, 0.8)
for (res in resolutions) {
  seurat_merged <- FindClusters(
    seurat_merged,
    resolution   = res,
    cluster.name = paste0("RNA_snn_res.", res)
  )
  cat(sprintf("Res %.1f: %d clusters\n", res, length(unique(seurat_merged@meta.data[[paste0("RNA_snn_res.", res)]])))  )
}

In [ ]:
# Resolution 비교 시각화
plot_list <- lapply(resolutions, function(res) {
  col_name <- paste0("RNA_snn_res.", res)
  DimPlot(seurat_merged, group.by = col_name, label = TRUE, label.size = 3) +
    ggtitle(paste0("Resolution ", res)) +
    NoLegend()
})

wrap_plots(plot_list, ncol = 2)

In [ ]:
# 최적 Resolution 선택 (세미나: 0.4 사용)
Idents(seurat_merged) <- "RNA_snn_res.0.4"
seurat_merged$seurat_clusters <- Idents(seurat_merged)

DimPlot(seurat_merged, reduction = "umap", label = TRUE, label.size = 4) +
  ggtitle("Final Clustering (Resolution 0.4)") +
  theme_minimal()

---
## Step 12. Marker Gene Identification (~15분)

### 🔍 FindAllMarkers

각 클러스터를 나머지 전체와 비교하여 **특이적으로 발현되는 유전자**를 찾습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_17.png" width="850"/>

*Fig. 13 — 세포 타입 어노테이션 전략: 자동 어노테이션 vs 수동 어노테이션 (Clake ZA et al., Nat Protoc 2019)*

In [ ]:
# 마커 유전자 탐색 (시간 소요: 5~15분)
# only.pos = TRUE: 해당 클러스터에서 높게 발현되는 유전자만
markers <- FindAllMarkers(
  seurat_merged,
  only.pos          = TRUE,
  min.pct           = 0.25,  # 최소 25% 세포에서 발현
  logfc.threshold   = 0.25   # 최소 log2FC 0.25
)

# Top 5 마커 확인
markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 5) %>%
  print(n = Inf)

In [ ]:
# Top 10 마커 히트맵
top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 10)

DoHeatmap(seurat_merged, features = top10$gene) +
  theme(axis.text.y = element_text(size = 6))

---
## Step 13. Cell Type Annotation (~20분)

### 망막 세포 타입 마커 (GSE210543 기반)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_18.png" width="850"/>

*Fig. 14 — 세포 타입 어노테이션 방법 및 Tip 4: 연구자의 주관이 중요!*

| 세포 타입 | 마커 유전자 |
|----------|------------|
| Retinal Ganglion Cells (RGC) | RBPMS, SNCG, GAP43 |
| Amacrine cells | GAD1, GAD2, TFAP2A |
| Bipolar cells | VSX2, CABP5, PRKCA |
| Müller glia | RLBP1, GLUL, SLC1A3 |
| Photoreceptors (Rod) | RHO, NR2E3 |
| Photoreceptors (Cone) | ARR3, GNGT2, OPN1LW |
| Horizontal cells | ONECUT2, LHX1 |
| Microglia | CX3CR1, P2RY12 |
| Endothelial cells | PECAM1, CDH5 |
| Progenitor cells | VSX2, FGF19, LIN28B |

> Tip: 발달기(Young) 샘플에는 Progenitor가 많고, Adult(Old)에는 분화된 세포가 많습니다.

In [ ]:
# 알려진 망막 마커 시각화
retinal_markers <- list(
  "Progenitor"   = c("VSX2", "FGF19", "LIN28B"),
  "RGC"          = c("RBPMS", "SNCG"),
  "Amacrine"     = c("GAD1", "TFAP2A"),
  "Bipolar"      = c("CABP5", "PRKCA"),
  "Muller_glia"  = c("RLBP1", "GLUL"),
  "Rod"          = c("RHO", "NR2E3"),
  "Cone"         = c("ARR3", "GNGT2"),
  "Horizontal"   = c("ONECUT2", "LHX1"),
  "Microglia"    = c("CX3CR1", "P2RY12")
)

all_markers <- unlist(retinal_markers)
# 데이터에 존재하는 유전자만 선택
valid_markers <- all_markers[all_markers %in% rownames(seurat_merged)]

DotPlot(seurat_merged, features = valid_markers, group.by = "seurat_clusters") +
  RotatedAxis() +
  ggtitle("Known Retinal Cell Type Markers") +
  theme(axis.text.x = element_text(size = 8))

In [ ]:
# 클러스터 → 세포 타입 매핑 (분석 결과 보고 수정)
# 아래는 예시 - 실제 마커 확인 후 조정 필요
cluster_annotations <- c(
  "0"  = "Muller_glia",
  "1"  = "Progenitor",
  "2"  = "Bipolar",
  "3"  = "RGC",
  "4"  = "Amacrine",
  "5"  = "Rod",
  "6"  = "Cone",
  "7"  = "Horizontal",
  "8"  = "Microglia",
  "9"  = "Unknown"
)

seurat_merged$cell_type <- plyr::mapvalues(
  as.character(seurat_merged$seurat_clusters),
  from = names(cluster_annotations),
  to   = cluster_annotations
)

# 최종 UMAP
DimPlot(seurat_merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, label.size = 3, repel = TRUE) +
  ggtitle("Cell Type Annotation") +
  theme_minimal()

In [ ]:
# Young vs Old: 세포 타입 구성 비교
prop_df <- seurat_merged@meta.data %>%
  group_by(group, cell_type) %>%
  summarise(n = n(), .groups = "drop") %>%
  group_by(group) %>%
  mutate(proportion = n / sum(n))

ggplot(prop_df, aes(x = group, y = proportion, fill = cell_type)) +
  geom_bar(stat = "identity") +
  scale_fill_brewer(palette = "Set3") +
  labs(title = "Cell Type Composition: Young vs Old",
       x = "Group", y = "Proportion") +
  theme_minimal()

In [ ]:
# 최종 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "02_seurat_annotated.rds"))
cat("Saved: 02_seurat_annotated.rds\n")

# 요약
cat("\n=== Final Summary ===\n")
cat("Total cells:", ncol(seurat_merged), "\n")
cat("Cell types:\n")
print(table(seurat_merged$cell_type, seurat_merged$group))

---
## ✅ Part 2 완료!

### 정리

| 단계 | 내용 |
|------|------|
| Cell Cycle | CellCycleScoring → ScaleData 회귀 |
| PCA | RunPCA (50 PC) → ElbowPlot으로 30 PC 선택 |
| Integration | Harmony (샘플 배치 효과 보정) |
| UMAP | RunUMAP (dims = 1:30) |
| Clustering | FindNeighbors + FindClusters (Res 0.4) |
| Marker | FindAllMarkers → DoHeatmap |
| Annotation | Known markers + manual annotation |

### 다음 파트 (Advanced)
**Part 3: CellChat & Monocle3**
→ 세포 간 통신 네트워크 + Pseudotime trajectory 분석

---
*GSE210543 | Human Retina scRNA-seq | KU Online Seminar*